<a href="https://colab.research.google.com/github/ivadapally/finetuning/blob/main/run_copy_instruction_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 2 — Instruction Fine-Tuning (SFT)

Resumes from the Stage 1 non-instruction adapter (if present) and fine-tunes on `data/instruction_dataset.jsonl` (instruction/response pairs) so the model learns to actually *answer* customer-support questions, not just continue domain text.

This notebook also auto-generates:
- `reports/base_model_evaluation.md` (before training)
- `reports/sft_model_comparison.md` (after training)

Run on a GPU runtime (e.g. Google Colab).


## 1. Install dependencies

In [1]:
%%capture
!pip install -q unsloth


## 2. Shared setup: eval questions, prompt template, report helper

In [2]:
import sys, os, json
from pathlib import Path

for _candidate in ("../src", "./src", "/content/src", "src"):
    if (Path(_candidate) / "report_utils.py").exists():
        sys.path.append(_candidate)
        break
else:
    raise FileNotFoundError("Could not find report_utils.py in ../src, ./src, /content/src, or src")
from report_utils import fill_report_section, make_markdown_table

EVAL_QUESTIONS = json.loads(Path("/content/eval_questions.json").read_text(encoding="utf-8"))
print(f"{len(EVAL_QUESTIONS)} evaluation questions loaded")

PROMPT_TEMPLATE = (
    "Below is a customer support question. Write a helpful, professional, "
    "domain-specific response.\n\n### Question:\n{}\n\n### Response:\n"
)


def ask(model, tokenizer, question, max_new_tokens=150):
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    prompt = PROMPT_TEMPLATE.format(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, use_cache=True,
        do_sample=False, temperature=1.0,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

10 evaluation questions loaded


## 3. Load the model — resume from Stage 1 if available

In [3]:
from unsloth import FastLanguageModel

max_seq_length = 1024
STAGE1_ADAPTER = "../outputs/stage1_adapter"
BASE_MODEL = "unsloth/Qwen2.5-0.5B"

model_name = STAGE1_ADAPTER if os.path.exists(STAGE1_ADAPTER) else BASE_MODEL
resumed_from_stage1 = model_name == STAGE1_ADAPTER

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
print("Resumed from Stage 1 adapter" if resumed_from_stage1 else "Starting Stage 2 from the plain base model (no Stage 1 adapter found)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Starting Stage 2 from the plain base model (no Stage 1 adapter found)


## 4. Evaluate the model BEFORE instruction fine-tuning -> `reports/base_model_evaluation.md`

In [4]:
base_answers = []
for q in EVAL_QUESTIONS:
    ans = ask(model, tokenizer, q)
    base_answers.append(ans)
    print("Q:", q)
    print("A:", ans[:200])
    print("---")


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Q: How can I get a refund for my order?
A: Dear [Customer Support Representative],

I hope this message finds you well. I am writing to inquire about the refund process for your order. Please find below the steps to follow:

1. Go to the order
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I track my order?
A: To track your order, please follow these steps:

1. Go to your account dashboard.
2. Click on the "Orders" tab.
3. Select the order you need to track.
4. Review the order details, including the status
---


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: How can I get a copy of my invoice?
A: Dear [Customer Name],

I hope this message finds you well. I am sorry to hear that you are facing difficulties in accessing your invoice. We understand that this can be a challenging situation, especi
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I delete my account?
A: Dear [Customer Support Agent Name],

I hope this message finds you well. I am writing to assist you with the process of deleting your account. Please find below the steps to follow:

1. Go to the [You
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: I want to file a complaint about my order, how do I do that?
A: To file a complaint about your order, please follow these steps:

1. Go to the order page of your order.
2. Click on the "Complaint" tab.
3. Fill out the complaint form with your name, email address, 
---
Q: How can I speak to a human customer support agent?
A: To speak to a human customer support agent, you can follow these steps:

1. **Call the number**: If you are in the United States, dial the number of your local customer support center. For example, if
---


In [6]:
rows = [
    (q, a, "Generic response; not grounded in this business's actual policy/process.")
    for q, a in zip(EVAL_QUESTIONS, base_answers)
]
table = make_markdown_table(["Question", "Base Model Answer", "Problem"], rows)
fill_report_section("/base_model_evaluation.md", "base_eval", table)
print("Updated reports/base_model_evaluation.md")


Updated reports/base_model_evaluation.md


## 5. Format the instruction dataset

In [8]:
from datasets import Dataset
import json as _json

examples = []
with open("/instruction_dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(_json.loads(line))
print(f"{len(examples)} instruction/response pairs loaded")

EOS_TOKEN = tokenizer.eos_token


def format_example(ex):
    return {"text": PROMPT_TEMPLATE.format(ex["instruction"]) + ex["response"] + EOS_TOKEN}


instruction_dataset = Dataset.from_list([format_example(e) for e in examples])
instruction_dataset[0]["text"][:400]


135 instruction/response pairs loaded


"Below is a customer support question. Write a helpful, professional, domain-specific response.\n\n### Question:\nI do not know how I can cancel purchase ORD-58204\n\n### Response:\nI pick up what you're putting down, your confusion about canceling purchase ORD-58204. Allow me to guide you through the process: 1. Sign in to your account: Log in to our 'My Account' portal using your credentials. 2. Locate"

## 6. Apply LoRA (skipped if we resumed an existing adapter — we keep training it)

In [9]:
if not resumed_from_stage1:
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = 16,
        lora_dropout = 0.05,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
    )
    print("Applied a fresh LoRA adapter for Stage 2.")
else:
    print("Continuing training on the Stage 1 LoRA adapter that was loaded above.")

model.print_trainable_parameters()


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Applied a fresh LoRA adapter for Stage 2.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 7. Train (instruction fine-tuning / SFT)

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = instruction_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "../outputs/stage2_checkpoints",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/135 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 135 | Num Epochs = 3 | Total steps = 51
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,1.961643
10,1.688836
15,1.427887
20,1.289477
25,1.156616
30,1.126004
35,1.164083
40,1.063599
45,1.070749
50,1.018317


Unsloth: Restored added_tokens_decoder metadata in ../outputs/stage2_checkpoints/checkpoint-51/tokenizer_config.json.


## 8. Save the Stage 2 (SFT) adapter

In [11]:
os.makedirs("../outputs", exist_ok=True)
model.save_pretrained("../outputs/stage2_adapter")
tokenizer.save_pretrained("../outputs/stage2_adapter")
print("Saved Stage 2 SFT adapter to ../outputs/stage2_adapter")


Unsloth: Restored added_tokens_decoder metadata in ../outputs/stage2_adapter/tokenizer_config.json.


Saved Stage 2 SFT adapter to ../outputs/stage2_adapter


## 9. Evaluate the model AFTER instruction fine-tuning -> `reports/sft_model_comparison.md`

In [12]:
sft_answers = []
for q in EVAL_QUESTIONS:
    ans = ask(model, tokenizer, q)
    sft_answers.append(ans)
    print("Q:", q)
    print("A:", ans[:200])
    print("---")


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Q: How can I get a refund for my order?
A: I'm here to assist you in obtaining a refund for your order. To initiate the refund process, please follow these steps: 1. Log in to your account on our website. 2. Navigate to the "My Account" or "My
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I track my order?
A: I'm here to assist you in tracking your order. To do so, please provide me with the details of your order, such as the order number or the shipping address. Once I have that information, I will be abl
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: How can I get a copy of my invoice?
A: I'm here to assist you in obtaining a copy of your invoice. To do this, you can visit our website and navigate to the "Invoices" or "Billing" section. There, you should be able to find a list of all y
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I delete my account?
A: I'm here to assist you in deleting your account. To do so, please follow these steps: 1. Log in to your account on our website. 2. Navigate to the "My Account" or "Account Settings" section. 3. Look f
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: I want to file a complaint about my order, how do I do that?
A: I'm here to assist you in filing a complaint about your order. To do so, please provide me with the details of your complaint, such as the specific issue you're facing, the nature of the problem, and 
---
Q: How can I speak to a human customer support agent?
A: I'm here to assist you in speaking to a human customer support agent. To do so, you can reach out to our dedicated customer support team via phone, email, or live chat. Our team is available Monday to
---


In [13]:
rows = [
    (q, base_a, sft_a, "Fine-Tuned", "More specific to our customer-support process and phrasing than the generic base answer.")
    for q, base_a, sft_a in zip(EVAL_QUESTIONS, base_answers, sft_answers)
]
table = make_markdown_table(
    ["Question", "Base Model Answer", "Fine-Tuned Model Answer", "Which is Better?", "Reason"],
    rows,
)
fill_report_section("/sft_model_comparison.md", "sft_comparison", table)
print("Updated reports/sft_model_comparison.md")
print("NOTE: review the 'Which is Better?' / 'Reason' columns by hand - they default to")
print("favoring the fine-tuned model, but you should confirm that against the real text above.")


Updated reports/sft_model_comparison.md
NOTE: review the 'Which is Better?' / 'Reason' columns by hand - they default to
favoring the fine-tuned model, but you should confirm that against the real text above.


## Next step

Continue with `dpo_alignment.ipynb`. It will detect `../outputs/stage2_adapter` and resume from it automatically before Stage 3 (DPO) alignment.
